In [0]:

import os

VARIAVEIS = ["S3_ENDPOINT", "AWS_DEFAULT_REGION", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"]

for nome in VARIAVEIS:
    if os.environ.get(nome):                       # já veio do job (Environment variables)? usa esse
        continue
    dbutils.widgets.text(nome, "")                 # senão, a caixinha (o job preenche pelos Parameters)
    os.environ[nome] = dbutils.widgets.get(nome)   # e o valor vira variável de ambiente

faltando = [nome for nome in VARIAVEIS if not os.environ[nome]]
assert not faltando, f"preencha: {faltando}"      # para aqui se faltar alguma chave

S3_BUCKET = "DataLakeVertice"
RAW = "/Volumes/lakehouse_vertice/bronze/volume_1"
print("ambiente pronto:", os.environ["S3_ENDPOINT"])



import os

VARIAVEIS = ["S3_ENDPOINT", "AWS_DEFAULT_REGION", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"]

for nome in VARIAVEIS:
    if os.environ.get(nome):                       # já veio do job (Environment variables)? usa esse
        continue
    dbutils.widgets.text(nome, "")                 # senão, a caixinha (o job preenche pelos Parameters)
    os.environ[nome] = dbutils.widgets.get(nome)   # e o valor vira variável de ambiente

faltando = [nome for nome in VARIAVEIS if not os.environ[nome]]
assert not faltando, f"preencha: {faltando}"      # para aqui se faltar alguma chave

S3_BUCKET = "DataLakeVertice"
RAW = "/Volumes/lakehouse_vertice/bronze/volume_1"
print("ambiente pronto:", os.environ["S3_ENDPOINT"])

# COMMAND ----------

# MAGIC %md
# MAGIC ## Passo 1 · Conectar no data lake
# MAGIC
# MAGIC Repare: nenhuma chave no código. O `boto3` lê `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` e
# MAGIC `AWS_DEFAULT_REGION` do ambiente; só o endereço do Supabase vai explícito.

# COMMAND ----------

import boto3
from botocore.config import Config

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["S3_ENDPOINT"],
    config=Config(s3={"addressing_style": "path"}),   # o Supabase só entende o bucket no caminho da URL
)

for bucket in s3.list_buckets()["Buckets"]:
    print(bucket["Name"])

# COMMAND ----------

# MAGIC %md
# MAGIC ## Passo 2 · O que tem no bucket?
# MAGIC
# MAGIC Cada arquivo é um objeto com uma **Key** (o caminho) e um **Size**. O S3 devolve no máximo 1.000
# MAGIC por página; o *paginator* busca todas as páginas.

# COMMAND ----------

objetos = []
for pagina in s3.get_paginator("list_objects_v2").paginate(Bucket=S3_BUCKET):
    objetos += pagina.get("Contents", [])

# só arquivos (o storage cria um objeto vazio de marcação em cada pasta)
arquivos = [o["Key"] for o in objetos if not o["Key"].endswith("/") and o["Size"] > 0
            and not o["Key"].endswith(".emptyFolderPlaceholder")]

from collections import Counter
for pasta, n in sorted(Counter(k.split("/")[0] for k in arquivos).items()):
    print(f"{pasta:<12} {n:>3} arquivos")
print(f"{'total':<12} {len(arquivos):>3}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Passo 3 · Apague o trabalho manual de ontem
# MAGIC
# MAGIC É proposital: as pastas do raw e as 9 tabelas da bronze somem, e voltam vindas do data lake.

# COMMAND ----------

TABELAS = ["projetos", "status_semanal", "excel_tarefas", "clickup_tarefas", "msproject_tarefas",
           "colaboradores", "apontamentos", "custos", "entregas_fornecedor"]

for tabela in TABELAS:
    spark.sql(f"DROP TABLE IF EXISTS lakehouse_vertice.bronze.{tabela}") 

for pasta in sorted({k.split("/")[0] for k in arquivos}):
    dbutils.fs.rm(f"{RAW}/{pasta}", True)
    print("apagada:", f"{RAW}/{pasta}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Passo 4 · Extrair e pousar: cada arquivo do bucket vai para o raw

# COMMAND ----------

import os

def baixar(key):
    destino = f"{RAW}/{key}"                                   # o mesmo caminho do bucket
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    conteudo = s3.get_object(Bucket=S3_BUCKET, Key=key)["Body"].read()   # 1 · extrair
    with open(destino, "wb") as f:                                        # 2 · pousar
        f.write(conteudo)
    return len(conteudo)

total = 0
for key in arquivos:
    total += baixar(key)
print(f"{len(arquivos)} arquivos, {total/1e6:.1f} MB no raw")

ambiente pronto: https://ssiyltdsapfnogwdtjpz.storage.supabase.co/storage/v1/s3
ambiente pronto: https://ssiyltdsapfnogwdtjpz.storage.supabase.co/storage/v1/s3
DataLakeVertice
clickup       25 arquivos
compras        1 arquivos
documentos     9 arquivos
excel         20 arquivos
financeiro     1 arquivos
msproject     15 arquivos
pmo            2 arquivos
rh             1 arquivos
timesheet      1 arquivos
total         75
apagada: /Volumes/lakehouse_vertice/bronze/volume_1/clickup
apagada: /Volumes/lakehouse_vertice/bronze/volume_1/compras
apagada: /Volumes/lakehouse_vertice/bronze/volume_1/documentos
apagada: /Volumes/lakehouse_vertice/bronze/volume_1/excel
apagada: /Volumes/lakehouse_vertice/bronze/volume_1/financeiro
apagada: /Volumes/lakehouse_vertice/bronze/volume_1/msproject
apagada: /Volumes/lakehouse_vertice/bronze/volume_1/pmo
apagada: /Volumes/lakehouse_vertice/bronze/volume_1/rh
apagada: /Volumes/lakehouse_vertice/bronze/volume_1/timesheet
75 arquivos, 6.9 MB no raw


## Passo 5 · Carregar: o raw vira a bronze

Tudo como texto, sem limpar nada: a bronze guarda o que chegou. A limpeza é na silver.

In [ ]:
%sql
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.projetos AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/volume_1/pmo/projetos.csv',
                format => 'csv', header => true, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.status_semanal AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/volume_1/pmo/status_semanal.csv',
                format => 'csv', header => true, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.colaboradores AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/volume_1/rh/colaboradores.csv',
                format => 'csv', header => true, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.apontamentos AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/volume_1/timesheet/apontamentos.csv',
                format => 'csv', header => true, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.custos AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/volume_1/financeiro/custos.csv',
                format => 'csv', header => true, inferColumnTypes => false);

CREATE OR REPLACE TABLE lakehouse_vertice.bronze.entregas_fornecedor AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse_vertice/bronze/volume_1/compras/entregas_fornecedor.csv',
                format => 'csv', header => true, inferColumnTypes => false);


In [ ]:
%sql
-- as 20 planilhas: renomeia as colunas para snake_case e guarda o nome do arquivo (o de-para)
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.excel_tarefas AS
SELECT `ID`               AS id,
       `Fase`             AS fase,
       `Tarefa`           AS tarefa,
       `Responsável`      AS responsavel,
       `Início previsto`  AS inicio_previsto,
       `Término previsto` AS termino_previsto,
       `Início real`      AS inicio_real,
       `Término real`     AS termino_real,
       `% Concluído`      AS pct_concluido,
       `Horas previstas`  AS horas_previstas,
       url_decode(_metadata.file_name) AS arquivo
FROM read_files('/Volumes/lakehouse_vertice/bronze/volume_1/excel/',
                format => 'excel', headerRows => 1, inferColumnTypes => false);

-- as 25 listas do ClickUp: uma linha por tarefa, com o id e o nome da lista
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.clickup_tarefas AS
SELECT list.id   AS list_id,
       list.name AS list_name,
       t.*,
       url_decode(_metadata.file_name) AS arquivo
FROM (
    SELECT list, explode(tasks) AS t, _metadata
    FROM read_files('/Volumes/lakehouse_vertice/bronze/volume_1/clickup/',
                    format => 'json', multiLine => true)
);

-- os 15 XML do MS Project: uma linha por <Task>
CREATE OR REPLACE TABLE lakehouse_vertice.bronze.msproject_tarefas AS
SELECT * EXCEPT (_rescued_data),
       url_decode(_metadata.file_name) AS arquivo
FROM read_files('/Volumes/lakehouse_vertice/bronze/volume_1/msproject/',
                format => 'xml', rowTag => 'Task');
